In [35]:
#Exercise 2
#Question 5
using GLPK, JuMP

J = 1:15                  # initiatives
T = 1:3                   # teams

b = [12, 24, 20, 28, 10, 11, 30, 27, 22, 25, 40, 38, 32, 34, 36]        # Strategic benefit
a = [14,  8,  7, 10, 13, 12,  9,  8,  7,  8, 12, 11, 10,  9, 10]               # Resource consumption
C = [28, 32, 35]    # Team capacities

# Dependency pairs:
dependencies = [    # (i,j) means initiative i requires initiative j
    (2,1),
    (3,1),
    (4,1),
    (7,5),
    (8,5),
    (9,6),
    (10,6),
    (11,7),
    (12,8),
    (15,10)]

model = Model(GLPK.Optimizer)

@variable(model,x[T,J],Bin)
@objective(model, Max, sum( x[i,j] * b[j] for i=T,j=J))

@constraint(model, [j=J], sum( x[i,j] for i=T) <= 1)
@constraint(model, [i=T], sum( x[i,j] * a[j] for j=J) <= C[i])

for dep in dependencies
    @constraint(model, sum( x[i,dep[1]] for i=T)  <= sum( x[i,dep[2]] for i=T) )
end

@constraint(model, sum( x[i,11] + x[i,12] for i=T) <= 1)
@constraint(model, sum( x[i,2] + x[i,3] + x[i,4] + x[i,7] + x[i,8] for i=T ) >= 3)

optimize!(model)

println("solver status: ", termination_status(model))
println("solution status: ", primal_status(model))
println("object value: ", objective_value(model))

print("\n")
for i=T
    for j=J
        print(value(x[i,j]))
        print(" ")
    end
    print("\n")
end

solver status: OPTIMAL
solution status: FEASIBLE_POINT
object value: 244.0

0.0 1.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 1.0 
1.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 0.0 
0.0 0.0 1.0 0.0 0.0 1.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 1.0 0.0 


In [ ]:
#Exercise 2
#Question 6/7   (Lagrange Dual solution)
using GLPK, JuMP
J = 1:15                  # initiatives
T = 1:3                   # teams

b = [12, 24, 20, 28, 10, 11, 30, 27, 22, 25, 40, 38, 32, 34, 36]        # Strategic benefit
a = [14,  8,  7, 10, 13, 12,  9,  8,  7,  8, 12, 11, 10,  9, 10]               # Resource consumption
C = [28, 32, 35]    # Team capacities
lambda = [2.9,3.8,1.03]

# Dependency pairs:
dependencies = [    # (i,j) means initiative i requires initiative j
    (2,1),
    (3,1),
    (4,1),
    (7,5),
    (8,5),
    (9,6),
    (10,6),
    (11,7),
    (12,8),
    (15,10)]

lambda = zeros(3)
bestUpperBound = Inf
bestLambda = copy(lambda)
maxIter = 500 


for k=1:maxIter
    # OPTIMIZATION MODEL
    model = Model(GLPK.Optimizer)

    @variable(model,x[T,J],Bin)
    @objective(model, Max, 
        sum( x[i,j] * b[j] for i=T,j=J)
        +
        sum( lambda[i] * (C[i] - sum( x[i,j] * a[j] for j=J) ) for i=T))

    @constraint(model, [j=J], sum( x[i,j] for i=T) <= 1)
    # @constraint(model, [i=T], sum( x[i,j] * a[j] for j=J) <= C[i])

    for dep in dependencies
        @constraint(model, sum( x[i,dep[1]] for i=T)  <= sum( x[i,dep[2]] for i=T) )
    end

    @constraint(model, sum( x[i,11] + x[i,12] for i=T) <= 1)
    @constraint(model, sum( x[i,2] + x[i,3] + x[i,4] + x[i,7] + x[i,8] for i=T ) == 3)

    optimize!(model)

    # UPDATING UPPER BOUND and LAMBDA
    UB = objective_value(model)
    if UB < bestUpperBound
        bestUpperBound = UB
        bestLambda = copy(lambda)
    end

    # CALC NEW LAMBDA
    for i=T
        mu = 0.05/sqrt(k)
        lambda[i] = max(lambda[i]  -  mu * ( C[i] - sum( value(x[i,j]) * a[j] for j=J) ), 0 )
    end
    # println(
    #     "k = ", k,
    #     "   UB = ", round(UB,digits=2),
    #     "   best UB = ", round(bestUpperBound,digits=2),
    #     "   lambda = ", round.(lambda,digits=2))
end

println()
println("BestLagrangianUB = ", bestUpperBound)
println("BestLambda = ", bestLambda)


BestLagrangianUB = 244.478908235479
BestLambda = [2.5513228169473043, 2.551803927883396, 2.5485154926658335]


In [ ]:
#Exercise 2
#Question 8 - LP-Relaxation (positive real valued)
using GLPK, JuMP

J = 1:15                  # initiatives
T = 1:3                   # teams

b = [12, 24, 20, 28, 10, 11, 30, 27, 22, 25, 40, 38, 32, 34, 36]        # Strategic benefit
a = [14,  8,  7, 10, 13, 12,  9,  8,  7,  8, 12, 11, 10,  9, 10]               # Resource consumption
C = [28, 32, 35]    # Team capacities

# Dependency pairs:
dependencies = [    # (i,j) means initiative i requires initiative j
    (2,1),
    (3,1),
    (4,1),
    (7,5),
    (8,5),
    (9,6),
    (10,6),
    (11,7),
    (12,8),
    (15,10)]

model = Model(GLPK.Optimizer)

@variable(model,x[T,J]>=0)
@objective(model, Max, sum( x[i,j] * b[j] for i=T,j=J))

@constraint(model, [j=J], sum( x[i,j] for i=T) <= 1)
@constraint(model, [i=T], sum( x[i,j] * a[j] for j=J) <= C[i])

for dep in dependencies
    @constraint(model, sum( x[i,dep[1]] for i=T)  <= sum( x[i,dep[2]] for i=T) )
end

@constraint(model, sum( x[i,11] + x[i,12] for i=T) <= 1)
@constraint(model, sum( x[i,2] + x[i,3] + x[i,4] + x[i,7] + x[i,8] for i=T ) >= 3)

optimize!(model)

println("solver status: ", termination_status(model))
println("solution status: ", primal_status(model))
println("object value: ", objective_value(model))

print("\n")
for i=T
    for j=J
        print(value(x[i,j]))
        print(" ")
    end
    print("\n")
end

solver status: OPTIMAL
solution status: FEASIBLE_POINT
object value: 254.89189189189193

0.33333333333333315 0.33333333333333315 0.33333333333333315 0.0 0.7817047817047819 0.0 0.0 1.0000000000000004 0.0 0.0 6.504336911945362e-17 0.015561015561015568 0.0 0.0 0.0 
0.0 0.0 0.0 0.33333333333333315 0.0 0.5945945945945946 0.0 0.0 0.0 0.5945945945945946 0.0 0.9844389844389849 0.0 0.0 0.5945945945945946 
0.0 0.0 0.0 0.0 0.21829521829521836 0.0 1.0000000000000002 0.0 0.5945945945945946 0.0 0.0 0.0 1.0 1.0 0.0 
